# dissim_1

In [707]:
import math
import numpy as np
import pandas as pd

In [708]:
# 数据格式如下
# disease = ["Meningitis, Pneumococcal", "Meningitis, Pneumococcal", "Mycetoma", "Botulism", "Botulism2", "Botulism3"]
# id = ["C01.252.200.500.600", "C08.345.654.570", "C01.252.410.040.692.606", "C01.252.410.222.151", "C01.252.410.222.151", "C03.252.410.222.151"]

print("开始读取数据....")
# 读取数据
meshid = pd.read_csv('mesh_2024_C.csv', header=0)
disease = meshid['disease'].tolist()
id = meshid['ID'].tolist()

meshdis = pd.read_csv('mirBase_lunwen_01_disease_final.csv', header=0)
unique_disease = meshdis['C1'].tolist()

# 初始化字典，有重复也没关系
for i in range(len(disease)):
    disease[i] = {}

print("开始计算每个病的DV")
# 计算每个病的DV，又重复也没关系，之后再合并

#   从ID的末尾开始，逐步往前递归计算贡献度，
for i in range(len(disease)):

    if len(id[i]) > 3:
        disease[i][id[i]] = 1
        id[i] = id[i][:-4]
        # print(disease[i])
        if len(id[i]) > 3:
            disease[i][id[i]] = round(1 * 0.8, 5)
            id[i] = id[i][:-4]
            # print(disease[i])
            if len(id[i]) > 3:
                disease[i][id[i]] = round(1 * 0.8 * 0.8, 5)
                id[i] = id[i][:-4]
                # print(disease[i])
                if len(id[i]) > 3:
                    disease[i][id[i]] = round(1 * 0.8 * 0.8 * 0.8, 5)
                    id[i] = id[i][:-4]
                    # print(disease[i])
                    if len(id[i]) > 3:
                        disease[i][id[i]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                        id[i] = id[i][:-4]
                        # print(disease[i])
                        if len(id[i]) > 3:
                            disease[i][id[i]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                            id[i] = id[i][:-4]
                            # print(disease[i])
                            if len(id[i]) > 3:
                                disease[i][id[i]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                                id[i] = id[i][:-4]
                                # print(disease[i])
                                if len(id[i]) > 3:
                                    disease[i][id[i]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                                    id[i] = id[i][:-4]
                                    # print(disease[i])
                                else:
                                    disease[i][id[i][:3]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                                    # print(disease[i])
                            else:
                                disease[i][id[i][:3]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                                # print(disease[i])
                        else:
                            disease[i][id[i][:3]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                            # print(disease[i])
                    else:
                        disease[i][id[i][:3]] = round(1 * 0.8 * 0.8 * 0.8 * 0.8, 5)
                        # print(disease[i])
                else:
                    disease[i][id[i][:3]] = round(1 * 0.8 * 0.8 * 0.8, 5)
                    # print(disease[i])
            else:
                disease[i][id[i][:3]] = round(1 * 0.8 * 0.8, 5)
                # print(disease[i])
        else:
            disease[i][id[i][:3]] = round(1 * 0.8, 5)
            # print(disease[i])
    else:
        disease[i][id[i][:3]] = 1
        # print(disease[i])

#print("合并相同的病不同ID的DV")


#################################  疾病的语义值 semantic value of a disease ####################################
# 合并相同的病不同ID的DV

unique_disease = meshdis['C1'].tolist()

# 这个name用来判断
disease_name = meshid['disease'].tolist()
unique_disease_name = meshdis['C1'].tolist()

for i in range(len(unique_disease)):
    unique_disease[i] = {}
    for j in range(len(disease_name)):
        if unique_disease_name[i] == disease_name[j]:       # 如果当前疾病和MeSHID的疾病名称一样，则
            unique_disease[i].update(disease[j])        #   相当于把之前同一疾病所有的关联树都整合在了一起
    # print(unique_disease[i])



#################################  疾病的语义值 semantic value of a disease ####################################
similarity = np.zeros([len(unique_disease_name), len(unique_disease_name)])
# print(similarity)
print(similarity.shape)

print("计算相似度")
print(len(unique_disease_name))

for m in range(len(unique_disease_name)):
    for n in range(len(unique_disease_name)):
        denominator = sum(unique_disease[m].values()) + sum(unique_disease[n].values())     # 分母
        numerator = 0
        for k, v in unique_disease[m].items():  #   迭代疾病1的所有祖先节点    相同则相加
            if k in unique_disease[n].keys():   #   判断两种疾病的祖先节点是否一样，
                numerator += v + unique_disease[n].get(k)       # 分子    values相加
        if(denominator==0):
            if(m==n):
                similarity[m, n] =1
            else:
                similarity[m, n]=0
        else:
            similarity[m, n] = round(numerator/denominator, 5)

# print(similarity)
print("保存结果")

# 保存结果

result = pd.DataFrame(similarity)
result.to_csv('disSSim_1.csv',header=False,index = False)

开始读取数据....
开始计算每个病的DV
(246, 246)
计算相似度
246
保存结果


# dissim_2

In [709]:
# disease = ["Meningitis, Pneumococcal", "Meningitis, Pneumococcal", "Mycetoma", "Botulism", "Botulism2", "Botulism3"]
# id = ["C01.252.200.500.600", "C08.345.654.570", "C01.252.410.040.692.606", "C01.252.410.222.151", "C01.252.410.222.151", "C03.252.410.222.151"]

print("开始读取数据")
# 读取数据
meshid = pd.read_csv('mesh_2024_C.csv', header=0)
disease = meshid['disease'].tolist()
id = meshid['ID'].tolist()

meshdis = pd.read_csv('mirBase_lunwen_01_disease_final.csv', header=0)
unique_disease = meshdis['C1'].tolist()

# 先把各个病的整个家族储存到一个list中，把所有病储存到一个fullID中

disease_list = []
fullID = []

for i in range(len(id)):

    disease_family = [disease[i], id[i]]
    fullID.append(id[i])
    if len(id[i]) > 3:
        id[i] = id[i][:-4]
        disease_family.append(id[i])
        fullID.append(id[i])
        if len(id[i]) > 3:
            id[i] = id[i][:-4]
            disease_family.append(id[i])
            fullID.append(id[i])
            if len(id[i]) > 3:
                id[i] = id[i][:-4]
                disease_family.append(id[i])
                fullID.append(id[i])
                if len(id[i]) > 3:
                    id[i] = id[i][:-4]
                    disease_family.append(id[i])
                    fullID.append(id[i])
                    if len(id[i]) > 3:
                        id[i] = id[i][:-4]
                        disease_family.append(id[i])
                        fullID.append(id[i])
                        if len(id[i]) > 3:
                            id[i] = id[i][:-4]
                            disease_family.append(id[i])
                            fullID.append(id[i])
                            if len(id[i]) > 3:
                                id[i] = id[i][:-4]
                                disease_family.append(id[i])
                                fullID.append(id[i])
                                if len(id[i]) > 3:
                                    id[i] = id[i][:-4]
                                    disease_family.append(id[i])
                                    fullID.append(id[i])

    disease_list.append(disease_family)
id = meshid['ID'].tolist()


# 计算每个病的DV，用字典形式创建list

# 计算每个病在所有病中的出现次数,构建字典

# 方法一:
# 现在有fullID，用fullID和原始ID对比，对fullID中某一个ID，看有多少个原始ID包含它
# countID = []
#
# for i in range(len(fullID)):
#     target = fullID[i]
#     count = 0
#     for j in range(len(id)):
#         if target in id[j]:
#             count += 1
#     countID.append(count)
#
# print(countID)

# 方法二
# 直接统计fullID中每个ID的出现次数，因为每个病的集合中的元素都是唯一的，所以如果一个ID出现了，就证明这个ID在这个病中，所以一个ID出现多少次就证明有多少个病包含这个ID

disease_dv = {}
countdis = len(disease)

for key in fullID:
    disease_dv[key] = round(math.log((disease_dv.get(key, 0) + 1)/countdis, 10)*(-1), 5)

# print(disease_dv)   # 每个疾病及其祖先的贡献值列表  11648
# print('-' * 100, len(disease_dv))
#
# print(disease_list[-1])
# print(disease_list[-2])
# print('-' * 100, len(disease_list))        #   每个疾病词条的祖先拆分  为一列 11648
#
# print(fullID[-2])
# print('-'*100,len(fullID))         #   每个疾病词条拆分后的祖先  为一列    53593


id = meshid['ID'].tolist()
disease = meshid['disease'].tolist()

# 初始化字典，有重复也没关系
for i in range(len(disease)):
    disease[i] = {}

# 计算每个病的DV，又重复也没关系，之后再合并

for i in range(len(disease)):

    if len(id[i]) > 3:
        disease[i][id[i]] = disease_dv[id[i]]
        id[i] = id[i][:-4]
        # print(disease[i])
        if len(id[i]) > 3:
            disease[i][id[i]] = disease_dv[id[i]]
            id[i] = id[i][:-4]
            # print(disease[i])
            if len(id[i]) > 3:
                disease[i][id[i]] = disease_dv[id[i]]
                id[i] = id[i][:-4]
                # print(disease[i])
                if len(id[i]) > 3:
                    disease[i][id[i]] = disease_dv[id[i]]
                    id[i] = id[i][:-4]
                    # print(disease[i])
                    if len(id[i]) > 3:
                        disease[i][id[i]] = disease_dv[id[i]]
                        id[i] = id[i][:-4]
                        # print(disease[i])
                        if len(id[i]) > 3:
                            disease[i][id[i]] = disease_dv[id[i]]
                            id[i] = id[i][:-4]
                            # print(disease[i])
                            if len(id[i]) > 3:
                                disease[i][id[i]] = disease_dv[id[i]]
                                id[i] = id[i][:-4]
                                # print(disease[i])
                                if len(id[i]) > 3:
                                    disease[i][id[i]] = disease_dv[id[i]]
                                    id[i] = id[i][:-4]
                                    # print(disease[i])
                                else:
                                    disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                                    # print(disease[i])
                            else:
                                disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                                # print(disease[i])
                        else:
                            disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                            # print(disease[i])
                    else:
                        disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                        # print(disease[i])
                else:
                    disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                    # print(disease[i])
            else:
                disease[i][id[i][:3]] = disease_dv[id[i][:3]]
                # print(disease[i])
        else:
            disease[i][id[i][:3]] = disease_dv[id[i][:3]]
            # print(disease[i])
    else:
        disease[i][id[i][:3]] = disease_dv[id[i][:3]]
        # print(disease[i])

# print(disease)


# 合并相同的病不同ID的DV

unique_disease = meshdis['C1'].tolist()

# 这个name用来判断
disease_name = meshid['disease'].tolist()
unique_disease_name = meshdis['C1'].tolist()

for i in range(len(unique_disease)):
    unique_disease[i] = {}
    for j in range(len(disease_name)):
        if unique_disease_name[i] == disease_name[j]:
            unique_disease[i].update(disease[j])

# print(unique_disease)


similarity = np.zeros([len(unique_disease_name), len(unique_disease_name)])


for m in range(len(unique_disease_name)):
    for n in range(len(unique_disease_name)):
        denominator = sum(unique_disease[m].values()) + sum(unique_disease[n].values())
        numerator = 0
        for k, v in unique_disease[m].items():
            if k in unique_disease[n].keys():
                numerator += v + unique_disease[n].get(k)
        if (denominator == 0):
            if (m == n):
                similarity[m, n] = 1
            else:
                similarity[m, n] = 0
        else:
            similarity[m, n] = round(numerator/denominator, 5)

# print(similarity)
print("保存结果")
# 保存结果

result = pd.DataFrame(similarity)
result.to_csv('disSSim_2.csv',header=False,index = False)


开始读取数据
保存结果


# 合并两个疾病

In [710]:
# 读取数据

similarity1 = pd.read_csv('disSSim_1.csv', header=None, encoding='gb18030').values
similarity2 = pd.read_csv('disSSim_2.csv', header=None, encoding='gb18030').values

similarity1 = np.mat(similarity1)
similarity2 = np.mat(similarity2)


print(similarity1.shape)
# 矩阵尺寸为88行88列

disSSim=similarity1

for m in range(disSSim.shape[0]):
    for n in range(disSSim.shape[1]):
        # if(similarity1[m, n]==0 and similarity2[m, n]!=0):
        #     print("这不可能")
        disSSim[m, n] = (similarity1[m, n] + similarity2[m, n])*0.5
        if m == n:
            disSSim[m, n] = 0.8


# 保存结果

result = pd.DataFrame(disSSim)
result.to_csv('disSSim.csv',header=False,index = False)

(246, 246)


# miFunSim

In [ ]:
def miRNASS(md_adjmat, disease_ss, *args, **kwargs):

    rows = np.size(md_adjmat, 0)    # should be 585
    result = np.zeros((rows, rows))

    for i in range(rows):
        idx = [idx for (idx, val) in enumerate(md_adjmat[i,:]) if val ==1]      #   找到关联矩阵中的 1
        if idx==False:
            continue
        for j in range(i):
            idy = [idy for (idy, val) in enumerate(md_adjmat[j, :]) if val == 1]    #    返回第j行的列索引
            if idy==False:
                continue

            sum1 = 0
            sum2 = 0

            for k1 in range(len(idx)):
                temp_max=0
                for d1 in range(len(idy)):
                    if disease_ss[idx[k1], idy[d1]]>temp_max:
                        temp_max=disease_ss[idx[k1], idy[d1]]
                sum1 = sum1 + temp_max  #   max(disease_ss(idx[k1], idy))


            for k2 in range(len(idy)):
                temp_max = 0
                for d2 in range(len(idx)):
                    if disease_ss[idy[k2], idx[d2]] > temp_max:
                        temp_max = disease_ss[idy[k2], idx[d2]]
                sum2 = sum2 + temp_max
            result[i, j] = (sum1 + sum2) / len(idx) + len(idy)
            result[j, i] = result[i, j]
        for k in range(rows):
            result[k, k] = 1
    return result

def normalize(arr,maxx,minn):
    arr1=np.copy(arr)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            arr1[i][j] = (arr[i][j] -minn) / (maxx -minn)
        arr1[i][i]=1
    return arr1

def maxmin(A):
    maxx=-1000
    minn=1000
    for i in range (A.shape[0]):
        if max(A[i])>maxx:
            maxx=max(A[i])
        if min(A[i]) <minn:
            minn = min(A[i])
    return maxx, minn
print("基于dis综合（GIP and disSemSim）相似度计算circRNA语义相似度")
interactions_ori = pd.read_csv('mirBase_lunwen_101_MDA_.csv', sep=',', header=None)
disSim_ori = pd.read_csv('disSSim.csv', encoding='gb18030', header=None).values

miRNASS = miRNASS(np.array(interactions_ori),np.array(disSim_ori))


maxx,minn=maxmin(np.array(miRNASS))
miRNASS=normalize(np.array(miRNASS),maxx,minn)
result = pd.DataFrame(miRNASS)
result.to_csv('miFunSim_norm.csv',header=False,index = False)

# CosineSim

In [715]:
MDA_matrix = pd.read_csv('mirBase_lunwen_101_MDA_1.csv', sep=' ', header=None)

MDA_matrix=np.array(MDA_matrix)

print(np.array(MDA_matrix).shape)

# circRNA和diease的数量
nc=np.array(MDA_matrix).shape[0]
nd=np.array(MDA_matrix).shape[1]



## 计算circRNA之间的Cosine相似度

mirna_similarity= np.zeros([nc, nc])  # 行与行之间的相似度，初始化矩阵


for i in range(nc):
    for j in range(i):
        Denominator=(np.sum(MDA_matrix[i]**2)**0.5)*(np.sum(MDA_matrix[j]**2)**0.5) #计算cosine相似度公式的分母
        if (Denominator==0):
            mirna_similarity[i,j]=0
        else:
            mirna_similarity[i, j] = np.dot(MDA_matrix[i],MDA_matrix[j])/Denominator
        mirna_similarity[j,i]=mirna_similarity[i, j]
    mirna_similarity[i,i]=1

# 保存结果
result = pd.DataFrame(mirna_similarity)
result.to_csv('miCosSim.csv',header=False,index = False)






## 计算disease之间的Cosine相似度

MDA_matrix = MDA_matrix.T  # 转置方便计算

print(np.array(MDA_matrix).shape)

disease_similarity= np.zeros([nd, nd])  # 行与行之间的相似度，初始化矩阵


for i in range(nd):
    for j in range(i):
        Denominator=(np.sum(MDA_matrix[i]**2)**0.5)*(np.sum(MDA_matrix[j]**2)**0.5) #计算cosine相似度公式的分母
        if (Denominator==0):
            disease_similarity[i,j]=0
        else:
            disease_similarity[i, j] = np.dot(MDA_matrix[i],MDA_matrix[j])/Denominator
        disease_similarity[j,i]=disease_similarity[i, j]
    disease_similarity[i,i]=1
# 保存结果
result = pd.DataFrame(disease_similarity)
result.to_csv('disCosSim.csv',header=False,index = False)

(270, 246)
(246, 270)


# GipSim

In [716]:
MDA_matrix = pd.read_csv('mirBase_lunwen_101_MDA_1.csv', sep=' ', header=None)


MDA_matrix=np.array(MDA_matrix)

print(np.array(MDA_matrix).shape)

# circRNA和diease的数量
nc=np.array(MDA_matrix).shape[0]
nd=np.array(MDA_matrix).shape[1]



## 计算circRNA之间的高斯相互作用相似度

mirna_similarity= np.zeros([nc, nc])  # 行与行之间的相似度，初始化矩阵

normSum = 0
for i in range(nc):
    normSum += np.sum(MDA_matrix[i]**2)**0.5**2  # 按定义用二阶范数计算
print(normSum)

for i in range(nc):
    for j in range(nc):
        mirna_similarity[i, j] = math.exp((np.sum((CDA_matrix[i] - CDA_matrix[j])**2)**0.5**2)  * normSum/nc* (-1))
        if mirna_similarity[i, j] == 1:
            mirna_similarity[i, j] = 0.8  # 这里是一个大问题，两个向量相同可以说它有一定相关度，可是计算出相关度等于1又不合理，只能定义一个值

# 保存结果
result = pd.DataFrame(mirna_similarity)
result.to_csv('miGIPSim.csv',header=False,index=False)
# 注意，这样保存之后会多了一行一列行号序号，需要删除






## 计算disease之间的高斯相互作用相似度

MDA_matrix = MDA_matrix.T  # 转置方便计算

print(np.array(MDA_matrix).shape)

diease_similarity= np.zeros([nd, nd])  # 行与行之间的相似度，初始化矩阵

normSum = 0
for i in range(nd):
    normSum += np.sum(CDA_matrix[i]**2)**0.5**2  # 按定义用二阶范数计算
print(normSum)

for i in range(nd):
    for j in range(nd):
        diease_similarity[i, j] = math.exp((np.sum((CDA_matrix[i] - CDA_matrix[j])**2)**0.5**2)  * normSum/ nd * (-1))
        if diease_similarity[i, j] == 1:
            diease_similarity[i, j] = 0.8  # 这里是一个大问题，两个向量相同可以说它有一定相关度，可是计算出相关度等于1又不合理，只能定义一个值

# 保存结果
result = pd.DataFrame(diease_similarity)
result.to_csv('disGIPSim.csv',header=False,index = False)
# 注意，这样保存之后会多了一行一列行号序号，需要删除

(270, 246)
424.9236390786442
(246, 270)
385.4176729421406
